# Monitoring — data quality and drift check

Lightweight checks run after each ingestion: (1) null counts and out-of-range values, and (2) a mean/std shift ('z-shift') on key numeric features between the training set and a recent batch. Cheap enough to run on every ingest as a trip-wire; not a substitute for a full statistical test (see design doc future work: PSI / KS-test).

In [1]:
import pandas as pd

TRAIN_PATH = '../data/processed/training_table.csv'
RECENT_PATH = '../data/raw/daily_2026_08_08.csv'  # <- change to check a different batch

NUMERIC_FEATURES_TO_WATCH = ['tenure', 'MonthlyCharges', 'TotalCharges']
DRIFT_Z_THRESHOLD = 0.5  # mean shift greater than 0.5 std devs -> flag


## Data quality check

In [2]:
def quality_check(df):
    issues = {}
    null_counts = df.isnull().sum()
    issues['null_counts'] = {c: int(v) for c, v in null_counts.items() if v > 0}
    if 'MonthlyCharges' in df.columns:
        oor = df[(df['MonthlyCharges'] < 0) | (df['MonthlyCharges'] > 1000)]
        issues['monthly_charges_out_of_range'] = int(len(oor))
    if 'tenure' in df.columns:
        oor_t = df[(df['tenure'] < 0) | (df['tenure'] > 100)]
        issues['tenure_out_of_range'] = int(len(oor_t))
    return issues


## Drift check

In [3]:
def drift_check(train_df, recent_df):
    drift_report = {}
    flagged = False
    for col in NUMERIC_FEATURES_TO_WATCH:
        train_vals = pd.to_numeric(train_df[col], errors='coerce').dropna()
        recent_vals = pd.to_numeric(recent_df[col], errors='coerce').dropna()
        train_mean, train_std = train_vals.mean(), train_vals.std()
        recent_mean = recent_vals.mean()
        z_shift = abs(recent_mean - train_mean) / train_std if train_std > 0 else 0.0
        is_flagged = z_shift > DRIFT_Z_THRESHOLD
        flagged = flagged or is_flagged
        drift_report[col] = {
            'train_mean': round(float(train_mean), 3), 'train_std': round(float(train_std), 3),
            'recent_mean': round(float(recent_mean), 3), 'z_shift': round(float(z_shift), 3),
            'flagged': bool(is_flagged),
        }
    return {'per_feature': drift_report, 'any_drift_flagged': bool(flagged)}


## Run

In [4]:
train_df = pd.read_csv(TRAIN_PATH)
recent_df = pd.read_csv(RECENT_PATH)

quality = quality_check(recent_df)
drift = drift_check(train_df, recent_df)

if drift['any_drift_flagged']:
    print('[WARNING] Drift detected in one or more features:')
    for feat, r in drift['per_feature'].items():
        if r['flagged']:
            print(f"  - {feat}: train_mean={r['train_mean']} recent_mean={r['recent_mean']} z_shift={r['z_shift']}")
else:
    print('[OK] No significant drift detected.')

if any(v > 0 for v in quality.get('null_counts', {}).values()):
    print(f"[WARNING] Null values found: {quality['null_counts']}")

import json
report = {'data_quality': quality, 'drift': drift}
print(json.dumps(report, indent=2))


[OK] No significant drift detected.
{
  "data_quality": {
    "null_counts": {},
    "monthly_charges_out_of_range": 0,
    "tenure_out_of_range": 0
  },
  "drift": {
    "per_feature": {
      "tenure": {
        "train_mean": 32.201,
        "train_std": 24.572,
        "recent_mean": 33.199,
        "z_shift": 0.041,
        "flagged": false
      },
      "MonthlyCharges": {
        "train_mean": 64.98,
        "train_std": 29.987,
        "recent_mean": 63.622,
        "z_shift": 0.045,
        "flagged": false
      },
      "TotalCharges": {
        "train_mean": 2278.767,
        "train_std": 2268.412,
        "recent_mean": 2314.06,
        "z_shift": 0.016,
        "flagged": false
      }
    },
    "any_drift_flagged": false
  }
}


## Save report

In [5]:
import os
os.makedirs('../artifacts/eval', exist_ok=True)
with open('../artifacts/eval/drift_report_sample.json', 'w') as f:
    json.dump(report, f, indent=2)
